# Simulación de datos sintéticos (FinanceAI)

**1. Simulación** > 2. EDA > 3. Entrenamiento

Este cuaderno genera un conjunto de datos sintéticos autónomo, limpio y **estructuralmente coherente**. En lugar de asignar valores aleatorios ciegos, el perfil financiero de cada usuario (endeudamiento y ahorro) se calcula matemáticamente evaluando su historial anual de transacciones generadas. Esto garantiza una correlación lógica estricta para el entrenamiento de los algoritmos de Machine Learning, replicando el funcionamiento de un entorno de negocio real.

In [1]:
import pandas as pd
import numpy as np
import os
import json
from faker import Faker

# Fijar semilla de aleatoriedad para garantizar reproducibilidad total en el proyecto
SEED = 42
np.random.seed(SEED)
Faker.seed(SEED)
fake = Faker('es_ES')

## 1. Generación de usuarios

1800 usuarios con sus ingresos y datos de identidad base.

Previene la aparición de valores atípicos extremados (millonarios) que distorsionarían las proporciones de endeudamiento y la capacidad de ahorro en los cálculos posteriores.

In [2]:
n_usuarios = 1800

# Probando Faker para usar nombres realistas y únicos
nombres = [fake.first_name() for _ in range(n_usuarios)]

# Generación de ingresos mensuales ($1500 a $5000) con random.uniform
ingresos = np.round(np.random.uniform(1500, 5000, size=n_usuarios), 2)

df_usuarios = pd.DataFrame({
    'id': range(1, n_usuarios + 1),
    'nombre': nombres,
    'ingreso_mensual': ingresos
})

print(f"Check:\n{df_usuarios.tail().to_string(index=False)}")
print("- - -")
print("Estructura generada (instances, atributo):", df_usuarios.shape)

Check:
  id    nombre  ingreso_mensual
1796 Estefanía          1796.77
1797   Soledad          4007.13
1798     Chelo          1752.30
1799     Nacio          1749.40
1800   Yolanda          1542.38
- - -
Estructura generada (instances, atributo): (1800, 3)


## 2. Generación de transacciones
240k consumos asociados a los usuarios y limitados estrictamente a las 10 categorías definitivas.

In [3]:
n_transacciones = 240000

# 10 categorías acordadas + vocabulario enriquecido regional y genérico
diccionario_conceptos = {
    'Alimentacion': [
        '7-eleven', 'aki(corporación favorita)', 'alimentos congelados', 'alkosto', 'almacen de barrio',
        'almacen de comestibles', 'almacen express', 'almacenes iberia', 'alvi', 'amigo (walmart pr)',
        'ara', 'assaí atacadista', 'atacadão (grupo carrefour)', 'auto mercado', 'autoservicio de alimentos',
        'bodega aurrera (grupo walmart)', 'carniceria', 'carrefour', 'carrefour brasil', 'carrefour express',
        'carrefour maxi', 'carulla (grupo éxito)', 'central madeirense', 'changomás (grupo de narváez)', 'chedraui',
        'cienfuegos', 'circle k', 'compra de comida', 'compra de viveres', 'compra en carniceria',
        'compra en verduleria', 'compras de supermercado', 'costco', 'coto', 'd1',
        'despensa de don juan', 'despensa familiar', 'devoto', 'diarco', 'dietetica productos naturales',
        'disco', 'distribuidora de alimentos', 'día', 'econo', 'económico',
        'el rey', 'excelsior gama', 'extra', 'fiambreria y quesos', 'fidalga',
        'fortis', 'fresh fresh market', 'freshmart', 'frutas y verduras', 'fruteria',
        'heb', 'hipermaxi', 'hipermercado', 'hirota food express', 'isimo',
        'jumbo (cencosud)', 'jumbo (centro cuesta nacional - ccn)', 'ketal', 'kiosco y golosinas', 'kioscos yes',
        'komprão', 'la anónima', 'la colonia', 'la comer', 'la torre',
        'lacteos y embutidos', 'libertad', 'luvebras', 'líder', 'líder (walmart)',
        'líder express (walmart)', 'machetazo', 'makro', 'makro perú', 'mass',
        'masxmenos', 'maxi despensa', 'maxi palí', 'maxiconsumo', 'maxikioscos',
        'mayorista 10', 'mayorsa', 'megamaxi(corporación favorita)', 'megasuper', 'mercado carlos iii',
        'mercado central', 'mercado municipal', 'metro (cencosud)', 'mi comisariato (corporación el rosado)', 'minimarket 24hs',
        'mipymes', 'muffato', 'nacional (ccn)', 'ok market', 'ole',
        'olímpica', 'oxxo (femsa)', 'oxxo brasil', 'oxxo chile', 'paiz',
        'palí', 'panaderia artesanal', 'panamericana', 'pescaderia', 'plaza',
        'plaza lama', 'plaza vea y vivanda (inretail / intercorp)', 'polleria', 'pricesmart', 'pueblo supermarkets',
        'puma abarrotero', 'pão de açúcar', 'real', 'riba smith', 'rotiseria de comida',
        'sam\'s club', 'santa isabel', 'santa isabel (cencosud)', 'selectos', 'seven-eleven',
        'sirena (grupo ramos)', 'soriana', 'spid', 'stock', 'super 99',
        'super selectos', 'super xtra', 'supermax', 'supermaxi (corporación favorita)', 'supermercado',
        'supermercado express', 'supermercado mayorista', 'supermercados bravo', 'supermercados dragón de oro', 'supermercados la antorcha',
        'superseis', 'surtimax (grupo éxito)', 'tambo+', 'tata', 'tienda de abarrotes',
        'tienda inglesa', 'tiendas ara', 'tiendas trd caribe', 'to go stores', 'tottus (falabella)',
        'unimarc (smu)', 'va express', 'vea', 'verduleria', 'vital',
        'walmart costa rica', 'walmart de méxico', 'walmart el salvador', 'walmart guatemala', 'walmart honduras',
        'walmart nicaragua', 'wong (cencosud)', 'yaguar', 'zona sul', 'éxito (grupo éxito)',
    ],
    'Educacion': [
        'academia cotopaxi', 'academia de musica', 'academia del perpetuo socorro', 'american international school of bolivia', 'american nicaraguan school',
        'american school of asunción', 'american school of tegucigalpa', 'americanas (papelería)', 'antártica libros', 'arancel educativo',
        'arcega', 'ardisa', 'artemis edinter', 'baldwin school', 'belgrano day school',
        'blue valley school', 'bootcamp de tecnologia', 'capacitacion online', 'capacitacion profesional', 'carol morgan school',
        'casa cuesta', 'casa del escritor', 'casa norberto', 'centro cultural sampedrano', 'centro educativo bilingüe espíritu santo',
        'certificacion profesional', 'clases de apoyo', 'clases particulares', 'cogna (krotón)', 'colegio alcázares',
        'colegio alemán alexander von humboldt', 'colegio alemán de guatemala', 'colegio alemán de guayaquil', 'colegio alemán de la paz', 'colegio alemán de montevideo',
        'colegio alemán nicaragüense', 'colegio americano (asf)', 'colegio americano de guatemala', 'colegio americano de quito', 'colegio anglo colombiano',
        'colegio augusto walte', 'colegio bilingüe new horizons', 'colegio brader', 'colegio calasanz', 'colegio calvert',
        'colegio cardenal newman', 'colegio carol morgan', 'colegio carrasco', 'colegio centroamérica', 'colegio clara jackson de heber',
        'colegio cristo rey', 'colegio cristobal colón', 'colegio del sol', 'colegio don bosco', 'colegio emil friedman',
        'colegio evelyn rogers', 'colegio externado san josé', 'colegio fap', 'colegio gimnasio campestre', 'colegio goethe',
        'colegio humboldt', 'colegio integral el ávila', 'colegio interamericano', 'colegio internacional', 'colegio internacional de caracas',
        'colegio internacional montessori', 'colegio javier', 'colegio jefferson', 'colegio la salle', 'colegio lamatepec',
        'colegio lincoln', 'colegio los campitos', 'colegio los nogales', 'colegio lux mundi', 'colegio madrid',
        'colegio marista', 'colegio marista san josé', 'colegio marymount', 'colegio maya', 'colegio menor san francisco de quito',
        'colegio metodista', 'colegio nuestra señora de la merced', 'colegio nueva granada (cng)', 'colegio privado', 'colegio sagrados corazones recoleta',
        'colegio san agustín', 'colegio san antonio', 'colegio san carlos', 'colegio san ignacio de loyola', 'colegio san ignacio el bosque',
        'colegio san josé', 'colegio san martín de tours', 'colegio santa cecilia', 'colegio santa clara', 'colegio santa ursula',
        'colegio santiago de león de caracas', 'colegio sek ecuador', 'colegio seminario', 'colegio st. francis', 'colegio tabancura',
        'colegio teresiano', 'colegio valle verde', 'colégio anglo', 'colégio bandeirantes', 'colégio dante alighieri',
        'colégio maxi', 'colégio objetivo', 'colégio visconde de porto seguro', 'comercial papelera', 'compra de libros',
        'country day school', 'craighouse', 'crossroads christian academy', 'cuesta libros', 'cuota de posgrado',
        'cuota universitaria', 'curso de data science', 'curso de idiomas', 'cuspide libros', 'delcampo school',
        'derecho de examen', 'didáctica', 'dimeiggs', 'diplomatura universitaria', 'discovery school',
        'distal libros', 'distribuidora corripio', 'distribuidores de útiles locales', 'el hombre de la mancha', 'el lector',
        'el pensador', 'el sótano', 'entre páginas', 'escuela americana', 'escuela internacional de la habana',
        'escuela internacional sampedrana', 'escuela nacional de ballet fernando alonso', 'escuela seran', 'escuela vocacional de arte cubanacán', 'eton school',
        'feria chilena del libro', 'fgv (fundação getulio vargas)', 'fina selección', 'fotocopias y apuntes', 'gimnasio moderno',
        'gran morrison', 'greengates school', 'grupo ochoa', 'guarderia infantil', 'inkafarma (útiles básicos)',
        'innova schools', 'inscripcion a congreso', 'instituto cumbres', 'instituto de ensenanza', 'institutos pedagógicos superiores',
        'intec', 'international school of panama', 'jardin de infantes', 'juan marcet', 'kalunga',
        'la imprenta', 'la rebaja plus (útiles)', 'le biscuit (papelería y útiles)', 'lehmann', 'libreria escolar',
        'libreria universitaria', 'librería ateneo', 'librería claraluz', 'librería el péndulo', 'librería española',
        'librería fayad jamís', 'librería hispamer', 'librería intercontinental', 'librería internacional', 'librería internacional (habana vieja)',
        'librería la ceiba', 'librería la moderna poesía', 'librería la trinitaria', 'librería las novedades', 'librería latina',
        'librería lectura', 'librería linardi y risso', 'librería metodista', 'librería nacional', 'librería navarro',
        'librería norberto gonzález', 'librería pocho', 'librería san jerónimo', 'librería universal', 'librerías crisol',
        'librerías gandhi', 'librerías progreso', 'librerías yenny / el ateneo', 'libri mundi', 'liceo francés louis pasteur',
        'lincoln international academy', 'lincoln school', 'livraria da vila', 'livraria leitura', 'los amigos del libro',
        'lumen', 'lápiz lópez', 'marchand papelería', 'markham college', 'mateca',
        'material de estudio', 'matricula escolar', 'mosca', 'mr. books', 'national paper & supply',
        'newton college', 'nido de aguilas', 'northlands school', 'notre dame school', 'nova locus',
        'office 2000', 'office depot', 'office depot méxico', 'office max', 'officemax pr',
        'panama preparatory school', 'panamericana', 'papa & mamma', 'papelería ccc', 'papelería central',
        'papelería don bosco', 'papelería el centro', 'papelería pastorino', 'papelería san diego', 'papelería universal',
        'papelmar', 'pensi', 'pontificia universidad católica de chile (puc)', 'pontificia universidad católica de puerto rico', 'pontificia universidad católica del perú (pucp)',
        'pontificia universidad católica madre y maestra (pucmm)', 'pontifícia universidade católica (puc-sp / puc-rio)', 'porrúa', 'posgrado y maestria', 'preparatoria anáhuac',
        'puce (pontificia universidad católica del ecuador)', 'qué leo', 'redes anima educação', 'rodelag (útiles escolares)', 'roosvelt (american school)',
        'saint andrew’s school', 'saint george school', 'saint george\'s college', 'santiago college', 'saraiva',
        'seminario de capacitacion', 'servilibro', 'sistema bernoulli', 'sistema universitario ana g. méndez (uagm)', 'sophos',
        'st. brendan\'s school', 'st. george\'s college', 'staples argentina (officenet)', 'stella maris', 'tai heng',
        'tailoy', 'taller de arte', 'taller de programacion', 'taller de robotica', 'tecmilenio',
        'tecmilenio preparatoria', 'tecni-ciencia libros', 'tecnológico de monterrey (itesm)', 'the bookmark', 'the british schools',
        'the grange school', 'the reader\'s corner', 'tiendas de artículos escolares (estatales)', 'tiendas grafitti (papelería)', 'tony superpapelerías',
        'uade', 'udla (universidad de las américas)', 'uees (universidad espíritu santo)', 'unibe', 'unidad educativa bilingüe delta',
        'universal', 'universidad adolfo ibáñez (uai)', 'universidad americana', 'universidad americana (uam)', 'universidad anáhuac',
        'universidad apec (unapec)', 'universidad austral', 'universidad católica andrés bello (ucab)', 'universidad católica argentina (uca)', 'universidad católica boliviana san pablo (ucb)',
        'universidad católica de honduras (unicah)', 'universidad católica de la habana (seminario)', 'universidad católica del uruguay (ucu)', 'universidad católica nuestra señora de la asunción', 'universidad centroamericana "josé simeón cañas" (uca)',
        'universidad centroamericana (uca)', 'universidad columbia del paraguay', 'universidad de aquino bolivia (udabol)', 'universidad de belgrano (ub)', 'universidad de la empresa (ude)',
        'universidad de la sabana', 'universidad de lima', 'universidad de los andes', 'universidad de montevideo (um)', 'universidad de san andrés (udesa)',
        'universidad del desarrollo (udd)', 'universidad del istmo', 'universidad del norte', 'universidad del pacífico', 'universidad del rosario',
        'universidad del sagrado corazón', 'universidad del valle de guatemala (uvg)', 'universidad del valle de méxico (uvm)', 'universidad diego portales (udp)', 'universidad dr. josé matías delgado',
        'universidad eafit', 'universidad fidélitas', 'universidad francisco gavidia (ufg)', 'universidad francisco marroquín (ufm)', 'universidad iberoamericana (ibero)',
        'universidad interamericana de panamá', 'universidad interamericana de puerto rico', 'universidad internacional de las américas (uia)', 'universidad javeriana', 'universidad josé cecilio del valle (ujcv)',
        'universidad latina de costa rica', 'universidad latina de panamá', 'universidad mariano gálvez de guatemala (umg)', 'universidad metropolitana (unimet)', 'universidad metropolitana de honduras (umh)',
        'universidad monteávila', 'universidad nueva esparta', 'universidad nur', 'universidad ort uruguay', 'universidad panamericana (up)',
        'universidad peruana cayetano heredia', 'universidad peruana de ciencias aplicadas (upc)', 'universidad politécnica de nicaragua (upoli)', 'universidad privada boliviana (upb)', 'universidad rafael landívar (url)',
        'universidad santa maría la antigua (usma)', 'universidad tecnológica centroamericana (unitec)', 'universidad tecnológica de el salvador (utec)', 'universidad thomas more (utm)', 'universidad torcuato di tella (utdt)',
        'universidade presbiteriana mackenzie', 'usfq (universidad san francisco de quito)', 'utiles escolares', 'utilex', 'utilísima',
        'yduqs (estácio)', 'éxito (sección papelería)'
    ],
    'Electrodomesticos': [
        'abc din', 'aire acondicionado', 'alkomprar', 'alkosto', 'almacenes simán (sección)',
        'anthony’s', 'artefacta', 'ashley homestore (sección)', 'aspiradora robot', 'audiofoto',
        'auriculares inalambricos', 'bristol', 'cafetera express', 'camara fotografica', 'carlos gutiérrez',
        'casa nissei', 'casas bahia', 'celular smartphone', 'cetrogar', 'comercial pato',
        'comisariato de la construcción (comandato)', 'compra de heladera', 'compra de lavarropas', 'computadora de escritorio', 'coppel',
        'corripio', 'costco', 'crandon', 'curacao', 'de prati (sección)',
        'dismac', 'diunsa', 'divino', 'doit center (sección)', 'el gallo más gallo',
        'el machetazo', 'el palacio de hierro', 'electrónica panamericana', 'elektra', 'elektra (tiendas la curacao)',
        'estilos', 'estufa electrica', 'excelsior gama (sección)', 'extractor de aire', 'falabella',
        'falabella colombia (sección)', 'falabella retail', 'famsa', 'fast shop', 'freidora de aire',
        'frávega', 'félix b. maduro (sección)', 'gallo', 'garbarino', 'garzón (sección)',
        'geant (sección)', 'gollo', 'gonzález giménez', 'himalaya', 'hiraoka',
        'hites', 'horno electrico', 'importadora castro', 'impresora multifuncion', 'indurama (directo)',
        'ivoo', 'jcpenney (sección)', 'jetstereo', 'jumbo (cencosud - sección)', 'jumbo (sección)',
        'koper furniture (sección)', 'ktronix', 'la curacao', 'la polar', 'lady lee',
        'lavavajillas', 'licuadora y procesadora', 'linea blanca electrodomesticos', 'liverpool', 'lojas americanas (sección)',
        'líder (walmart)', 'magazine luiza', 'makro (sección)', 'marcimex', 'max',
        'megatone', 'mercado livre', 'microondas', 'minipimer', 'monge',
        'monitor gamer', 'motociclo (sección)', 'multi ahorro hogar (ta-ta)', 'musimundo', 'ngo (sección)',
        'notebook portatil', 'oechsle', 'olé (sección)', 'omnisport', 'paris (cencosud)',
        'parlante bluetooth', 'pava electrica', 'plancha de ropa', 'plaza lama (sección)', 'plaza vea (sección)',
        'ponto frio', 'prado', 'record electric', 'ribeiro', 'ripley',
        'rodéla', 'rooms to go (sección)', 'saga falabella (sección)', 'sam’s club', 'sears (sección)',
        'secador de pelo', 'shopping china (sección)', 'sinsa (sección)', 'smart tv', 'stevens',
        'suburbia (sección)', 'sukasa', 'tablet digital', 'televisor smart', 'tienda de electrodomesticos',
        'tienda naranja', 'tiendas caribe', 'tiendas checo', 'tiendas daka', 'tiendas panamericanas (cimex)',
        'tostadora electrica', 'tropigás', 'ventilador de techo', 'via varejo', 'vidri',
        'éxito (sección)'
    ],
    'Inversion': [
        'acciones (bbv)', 'acciones (bcv)', 'acciones (bvm)', 'acciones (bvn)', 'acciones (bvpasa)',
        'acciones (bvq / bvg)', 'acciones (bvrd)', 'acciones (latinex)', 'acciones de empresas listadas en la bolsa de valores de caracas', 'acciones y etfs (a través de brokers de ee. uu.)',
        'adquisicion de titulos publicos', 'ahorro mensual en dolares', 'al30', 'ambev (b3)', 'américa móvil (bmv / biva)',
        'aporte de capital', 'ações', 'banco de chile (bolsa de santiago)', 'banco inter', 'bancolombia (bvc)',
        'bcu / bcp', 'bold', 'bono cer (boncer / t2x)', 'bonos (bonos del tesoro, bonos corporativos)', 'bonos corporativos',
        'bonos de deuda pública (con alto riesgo y volatilidad)', 'bonos de la república (muy riesgosos/volátiles)', 'bonos de la república de cuba', 'bonos de la república de cuba (inversión estatal)', 'bonos del estado',
        'bonos del gobierno', 'bonos del gobierno (bnv)', 'bonos del gobierno (ministerio de hacienda)', 'bonos del tesoro', 'bonos del tesoro (letes)',
        'bonos del tesoro (letes/cetes)', 'bonos del tesoro (ministerio de hacienda)', 'bonos del tesoro (notas del tesoro)', 'bonos del tesoro (tes)', 'bonos del tesoro cubano',
        'bonos del tesoro público', 'bonos m', 'bonos municipales (bonos de pr)', 'bonos soberanos', 'bonos soberanos y de pdvsa (cotización limitada)',
        'buenaventura (bvl)', 'cajas municipales de ahorro y crédito (cmac)', 'cartera de inversiones', 'cauciones bursatiles', 'cd (certificates of deposit)',
        'cda (certificados de depósito de ahorro)', 'cdb', 'cdt (certificado de depósito a término)', 'cedes (certificados de depósito)', 'cemex (bmv / biva)',
        'certificados de depósito', 'certificados de depósito (cd)', 'certificados de depósito a plazo', 'certificados de depósito a plazo (cdp)', 'certificados de depósito a plazo fijo',
        'certificados de depósito a plazo fijo (cdp)', 'certificados de depósito a plazo fijo (en bancos estatales)', 'certificados de inversión', 'certificados del banco central (bcrd)', 'cetes',
        'compra bitcoin btc', 'compra de acciones', 'compra de cedears', 'compra de criptomonedas', 'compra de dolares',
        'compra usdt', 'constitucion plazo fijo', 'credicorp (bvl)', 'cuenta comitente broker', 'cuentas de ahorro a plazo fijo',
        'cuentas de ahorro en moneda libremente convertible (mlc)', 'dap', 'debêntures incentivadas', 'deposito en broker', 'depósitos a plazo fijo',
        'depósitos a plazo fijo (sujetos a alta inflación)', 'depósitos monetarios y de ahorro', 'dolar bolsa mep', 'dpf', 'ecopetrol (bvc)',
        'enel chile (bolsa de santiago)', 'fci money market', 'fci money market (t+0)', 'fci renta fija', 'fci renta fija (t+1 / t+2)',
        'fci renta variable', 'fic (fondos de inversión colectiva)', 'fintual', 'fondo comun de inversion', 'fondos de ahorro previsional (afaps)',
        'fondos de cesantía', 'fondos de inversión (operados por administradoras de fondos de inversión - afis)', 'fondos de inversión (operados por administradoras de fondos de inversión - afisa)', 'fondos de inversión (operados por bancos)', 'fondos de inversión (operados por gestoras de fondos)',
        'fondos de inversión (operados por puestos de bolsa)', 'fondos de inversión (operados por sociedades administradoras de fondos - safis)', 'fondos de inversión (operados por sociedades administradoras de inversión - sais)', 'fondos de inversión (sociedades de inversión)', 'fondos de inversión abiertos (fia)',
        'fondos de inversión cerrados (fic)', 'fondos de inversión de deuda', 'fondos de inversión gestionados por bancos', 'fondos de inversión variable', 'fondos de jubilaciones y pensiones',
        'fondos de pensiones', 'fondos de pensión complementaria', 'fondos mutuos (bbva asset management)', 'fondos mutuos (credicorp capital)', 'fondos mutuos (ffmm): balanceados',
        'fondos mutuos (ffmm): de acciones', 'fondos mutuos (ffmm): deuda nacional', 'fondos mutuos (interfondos)', 'fondos mutuos (us-based providers)', 'fundo di',
        'gd30', 'grupo financiero galicia (byma)', 'grupo méxico (bmv / biva)', 'inversion en cripto activos', 'inversion en fondos comunes',
        'inversion en renta variable', 'iras', 'isa (bvc)', 'itaú unibanco (b3)', 'lca (letras de crédito do agronegócio)',
        'lci (letras de crédito imobiliário)', 'lecaps / boncap', 'letras de regulación monetaria (lrm)', 'lulo bank', 'mercado pago',
        'mercado pago br', 'mercado pago cl', 'mercado pago co', 'mercado pago mx', 'mercado pago pe',
        'multimercado', 'máximo', 'naranja x', 'nu colombia', 'nu méxico (cajitas)',
        'nuconta (nubank)', 'obligaciones negociables', 'ons arcor', 'ons pampa', 'ons ypf',
        'operacion de cambio mep', 'pagaré con rendimiento liquidable al vencimiento (prlv)', 'pagarés bursátiles', 'pampa energía (byma)', 'papel comercial',
        'papeles comerciales de empresas locales', 'personal pay', 'petrobras (b3)', 'picpay', 'planes 401(k) y 403(b)',
        'plazo fijo', 'plazo fijo tradicional', 'plazo fijo uva', 'rendimiento de inversion', 'sofipos',
        'southern copper (bvl)', 'sqm (bolsa de santiago)', 'staking de cripto', 'stori', 'tenpo',
        'ternium (byma)', 'tes (títulos de tesorería del gobierno nacional)', 'tesouro direto (tesouro ipca+)', 'tesouro direto (tesouro pré-fixado)', 'tesouro direto (tesouro selic)',
        'transferencia a broker', 'títulos de deuda emitidos por empresas estatales', 'títulos de participación (en ciertos proyectos estatales o mixtos)', 'ualá', 'ualá mx',
        'udibonos', 'ui (unidades indizadas)', 'vale (b3)', 'walmex (bmv / biva)', 'yape',
        'ypf (byma)'
    ],
    'Ocio': [
        'adrián tropical', 'alba cinema', 'alquiler de cancha futbol', 'altice play', 'alto',
        'amazon prime video', 'andrés carne de res', 'antares', 'astrid y gastón', 'astrid y gastón paraguay',
        'auditorio luis elizondo', 'auditorio nacional adela reta', 'ayahuasca restobar', 'baar fun fun', 'bar da dona onça',
        'bar do juarez', 'bar inglés', 'bar nacional', 'barbarian', 'barra loka',
        'blim tv', 'bocana', 'bogotá beer company (bbc)', 'bonafide', 'boragó',
        'borola', 'bowling recreativo', 'brasil paralelo', 'cable onda go', 'cabletica play',
        'cacao de origen', 'cafeteria y merienda', 'cafeto', 'café barista', 'café britt',
        'café cuatro sombras', 'café de lima', 'café de olla', 'café devoción', 'café el escorial',
        'café la casona', 'café las flores', 'café magia', 'café martínez', 'café punta del cielo',
        'café república', 'café san alberto', 'café santo domingo', 'café tortoni', 'café unido',
        'cantina la ópera', 'caribbean cinemas', 'carmine gastronomía', 'carnaval bar', 'casa do pão de queijo',
        'casa k’ona', 'casco viejo (varios bares/restaurantes)', 'central', 'centro cultural helénico', 'centro de bellas artes luis a. ferré (santurce)',
        'centro deportivo', 'cerveceria artesanal', 'cervecería barbarian', 'cervecería kross (krossbar)', 'cervecería volcanes',
        'cervejaria colorado', 'chachingo', 'chao teatro', 'chapeau', 'cherusker (cervecería)',
        'chili’s nicaragua', 'ciclos café', 'cielito querido café', 'cine acapulco', 'cine center',
        'cine chaplin', 'cine star', 'cine yara', 'cine.ar play', 'cinema la pradera',
        'cinemark', 'cinemas galerías', 'cinemas intermezzo', 'cinemateca uruguaya', 'cines unidos',
        'cinex', 'cinépolis', 'claro tv+', 'claro video', 'claro video colombia',
        'claro video costa rica', 'claro video el salvador', 'claro video guatemala', 'claro video honduras', 'claro video nicaragua',
        'cnt play', 'coffee lab', 'complejo plaza', 'crepes & waffles', 'cubavisión internacional',
        'cuervo', 'cuota de gimnasio', 'd\'café', 'd.o.m.', 'directv go',
        'discoteca', 'disney+', 'don julio', 'el adobe', 'el arabito',
        'el batey', 'el borrego viudo', 'el conuco', 'el floridita', 'el garabato',
        'el gato negro', 'el goce pagano', 'el morito', 'el palenque', 'el patio',
        'el querandí', 'el zaguan', 'el ávila', 'entradas de cine', 'entradas de futbol',
        'entradas para concierto', 'entradas para recital', 'espectaculo de stand up', 'estación atocha', 'evento de entretenimiento',
        'fine arts cinema', 'flow', 'fonda lo que hay', 'fuente de soda (dominó)', 'gam (centro gabriela mistral)',
        'globoplay', 'gran rex', 'gran sala efraín recinos (centro cultural miguel ángel asturias)', 'gran teatro de la habana alicia alonso', 'gran teatro nacional',
        'grupocine', 'gustu', 'hacienda san pedro', 'handshake speakeasy', 'havanna',
        'hbo max', 'huerta bar', 'jacinto', 'jardín de asia', 'jose enrique',
        'juan valdez café', 'juan valdez chile', 'juegos de consola', 'kacao', 'karaoke y bar',
        'la bodeguita del medio', 'la cabrera', 'la casona del chef', 'la cava del duende', 'la central',
        'la factoría', 'la fonda', 'la pampa', 'la pampa argentina', 'la peninsular',
        'la pista', 'la placita de santurce (varios bares/restaurantes)', 'la rana dorada (brewery)', 'lattente', 'leo',
        'liberty everywhere', 'licorería limantour', 'lido bar', 'liguria', 'lo de osvaldo',
        'lo nuestro', 'los cocuyos', 'los planes de renderos (varios restaurantes)', 'lucia restaurante', 'maido',
        'maito', 'maute grill', 'mercado central', 'mercado de las brujas (comida callejera)', 'mercado de mariscos (comida callejera)',
        'mocotó', 'modesta', 'moreno', 'movistar play chile', 'movistar play colombia',
        'movistar play perú', 'multicine', 'márquez', 'nau lounge bar', 'netflix',
        'nscreen', 'nuevo mundo draught bar', 'obra de teatro', 'opera', 'origen tostadores de café',
        'p.f. changs ecuador', 'pago de delivery comida', 'palacio de bellas artes', 'palacio del cine', 'paladar la guarida',
        'paladar san cristóbal', 'palácio das artes (bh)', 'parador la huella', 'paramount+', 'park café',
        'parque acuatico', 'parque de atracciones', 'parque tematico', 'partido de paddle', 'paseo recreativo',
        'patio colombiano', 'persepolis', 'pira ró', 'pirajá', 'portofino',
        'pujol', 'pupuserías locales', 'quentin', 'quintonil', 'raíces',
        'rei do mate', 'rendibu', 'restaurante la casona', 'restaurante subanik', 'restaurante tin jo',
        'restaurante y bar', 'riivi', 'sala de escape', 'salida a boliche', 'salida a cenar',
        'saúl e. méndez', 'siete negronis', 'star+', 'starbucks mx', 'supercines',
        'suplicy cafés', 'suscripcion de musica', 'suscripcion de streaming', 'talleyrand', 'tap house',
        'teatro amazonas', 'teatro biobío (concepción)', 'teatro castro alves', 'teatro colón', 'teatro colón de bogotá',
        'teatro de la ciudad', 'teatro degollado', 'teatro del lago (frutillar)', 'teatro el círculo', 'teatro en círculo',
        'teatro general san martín', 'teatro gran mariscal de sucre (sucre)', 'teatro guaira', 'teatro heredia (adolfo mejía)', 'teatro independencia',
        'teatro insurgentes', 'teatro juárez', 'teatro la comedia', 'teatro mayor julio mario santo domingo', 'teatro melico salazar',
        'teatro metropolitano', 'teatro metropólitan', 'teatro municipal alberto saavedra pérez (la paz)', 'teatro municipal de arequipa', 'teatro municipal de caracas',
        'teatro municipal de lima', 'teatro municipal de santiago', 'teatro municipal de trujillo', 'teatro municipal de viña del mar', 'teatro municipal del cusco',
        'teatro municipal enrique buenaventura', 'teatro municipal ignacio a. pane', 'teatro nacional de costa rica', 'teatro nacional de panamá', 'teatro nacional de san salvador',
        'teatro nacional eduardo brito', 'teatro nacional manuel bonilla', 'teatro nacional rubén darío', 'teatro nacional sucre', 'teatro nescafé de las artes',
        'teatro pablo tobón uribe', 'teatro pirandello', 'teatro plaza', 'teatro principal', 'teatro renault',
        'teatro san martín (cba)', 'teatro san martín (tuc)', 'teatro solís', 'teatro sánchez aguilar', 'teatro teresa carreño',
        'teatro tpb', 'telecine play', 'temple bar', 'theatro municipal de são paulo', 'theatro municipal do rio de janeiro',
        'ticket de cine', 'tigo play', 'vera tv', 'vicios', 'videojuegos online',
        'villamorra cinecenter', 'vix', 'xtrim play', 'zapping tv', 'zazu',
    ],
    'Salud': [
        'aacd (associação de assistência à criança deficiente)', 'abc medical center', 'abono de obra social', 'amil diagnóstico', 'analisis de sangre',
        'apec (hospital de la ceguera)', 'ashford presbyterian community hospital', 'asociación española', 'asociación española (laboratorios/imágenes)', 'asociación española (servicios de rehabilitación)',
        'asociación española (servicios dentales)', 'asociación guatemalteca de esclerosis múltiple (agem)', 'atencion medica ambulatoria', 'atencion odontologica', 'atlantic dental group',
        'audifarma', 'bionet', 'bogotá láser', 'boticas perú', 'boticas y salud',
        'cafam droguerías', 'caribbean center for behavioral health', 'cemic (centro de educación médica e investigaciones clínicas)', 'centro cedimat', 'centro de diagnóstico integral (cdi)',
        'centro de diagnóstico médico (cedim)', 'centro de diagnóstico por imágenes idm', 'centro de especialidades odontológicas (ceo)', 'centro de imágenes médicas', 'centro de neurociencias integral (cenit)',
        'centro de neurorehabilitación y salud integral', 'centro de rehabilitación física (crf)', 'centro de rehabilitación física y terapia ocupacional', 'centro de rehabilitación integral (cri)', 'centro de rehabilitación integral dr. juan tanca marengo',
        'centro de rehabilitación psicosocial', 'centro de salud mental de la habana', 'centro de salud mental dr. francisco herrera luque', 'centro de visión láser', 'centro láser',
        'centro médico de puerto rico', 'centro médico la costa', 'centro médico paitilla', 'centro médico paitilla (laboratorios/imágenes)', 'centro médico paitilla (servicio de oftalmología)',
        'centro médico paitilla (servicios de psiquiatría)', 'centro neuropsiquiátrico', 'centro odontológico las mercedes', 'centro oftalmológico de nicaragua', 'centro oftalmológico de puerto rico',
        'centro oftalmológico dr. nano', 'centro oftalmológico integral (coi)', 'centro oftalmológico prof. daniel weil', 'centro oftalmológico santa lucía', 'centro rossi',
        'chopo diagnostic (imágenes)', 'christus muguerza', 'clinica privada', 'clofán', 'clínica abreu',
        'clínica abreu (servicios de salud mental)', 'clínica abreu (servicios dentales)', 'clínica abreu (servicios diagnósticos)', 'clínica alemana (servicios diagnósticos)', 'clínica alemana de santiago',
        'clínica americana (servicios diagnósticos)', 'clínica anglo americana', 'clínica barraquer', 'clínica biblica (servicio de odontología)', 'clínica bíblica (laboratorios/imágenes)',
        'clínica campestre (rehabilitación física)', 'clínica canto', 'clínica clofán', 'clínica corazones unidos', 'clínica cumbre',
        'clínica cuscatlán (servicios dentales)', 'clínica cuscatlán (servicios diagnósticos)', 'clínica de especialidades odontológicas (ceo)', 'clínica de la ansiedad', 'clínica de ojos caracas',
        'clínica de ojos de panamá', 'clínica de ojos dr. roberto solórzano', 'clínica de ojos dra. laura fleitas', 'clínica de ojos oftalmología', 'clínica de ojos santa cruz',
        'clínica de rehabilitación física', 'clínica de rehabilitación integral (cri)', 'clínica de trastornos de la alimentación (cta)', 'clínica delgado (auna)', 'clínica dental 26 de julio',
        'clínica dental americana', 'clínica dental dr. juan f. boscio', 'clínica dental dr. walter ochoa', 'clínica dental el marqués', 'clínica dental esthetic',
        'clínica dental paitilla', 'clínica dental palermo', 'clínica dental santa ana', 'clínica dental sonríe', 'clínica dental uno',
        'clínica dental white smile', 'clínica el cedral (salud mental)', 'clínica el sur (salud mental)', 'clínica el ávila', 'clínica estomatológica 10 de octubre',
        'clínica foianini', 'clínica foianini (laboratorio/imágenes)', 'clínica guayaquil', 'clínica hospital san fernando', 'clínica imbanaco',
        'clínica internacional', 'clínica internacional (servicios dentales)', 'clínica internacional (servicios diagnósticos)', 'clínica internacional cira garcía (diagnóstico)', 'clínica internacional varadero',
        'clínica kennedy', 'clínica kennedy (servicios dentales)', 'clínica kennedy (servicios diagnósticos)', 'clínica la inmaculada (salud mental)', 'clínica las condes',
        'clínica las condes (salud mental)', 'clínica los olivos', 'clínica maxilofacial', 'clínica monserrat (salud mental)', 'clínica odontológica central',
        'clínica odontológica del uruguay', 'clínica odontológica dentis', 'clínica odontológica equinoccial', 'clínica odontológica sanitas', 'clínica odontológica universidad de chile',
        'clínica oftalmológica andes', 'clínica oftalmológica cira garcía', 'clínica oftalmológica del caribe (coca)', 'clínica oftalmológica dr. juan zunino', 'clínica oftalmológica pasteur',
        'clínica oftalmológica san miguel', 'clínica oftalmológica visual láser', 'clínica ojos', 'clínica portoazul (auna)', 'clínica psiquiátrica universitaria (universidad de chile)',
        'clínica ricardo jiménez núñez (rehabilitación)', 'clínica ricardo palma', 'clínica ricardo palma (servicio de oftalmología)', 'clínica san juan de dios (rehabilitación)', 'clínica san pablo',
        'clínica san pablo (salud mental)', 'clínica santa lucía', 'clínica santa maría', 'clínica santa maría (banmédica)', 'clínica visualiza',
        'colmédica (diagnóstico por imágenes y laboratorios)', 'colmédica (red)', 'comisariato del ahorro', 'compra en farmacia', 'consulta dentista',
        'consulta dermatologica', 'consulta medica especialista', 'consulta medica general', 'consulta psicologo', 'consulta traumatologo',
        'corazones unidos (servicios diagnósticos)', 'crit (teletón)', 'cruz verde', 'cruz verde colombia', 'cuota medicina prepaga',
        'cvs pharmacy pr', 'dasa (delboni auriemo, sérgio franco, lavoisier)', 'dental associates psc', 'dental care panama', 'dental corominas',
        'dental home', 'dental unlimited', 'dentalia', 'dentimagen', 'dentis group',
        'dentistry for children', 'dentix', 'diagnostico por imagenes', 'diagnóstico 2000', 'diagnóstico maipú',
        'dra. maría mercedes', 'droga raia / drogasil (rd)', 'drogaria pacheco', 'drogaria são paulo', 'drogarias bifarma',
        'droguerías la rebaja', 'droguerías olímpica', 'ecografia medica', 'estudio de radiografia', 'estudio ecocardiograma',
        'estudios cardiologicos', 'estudios de laboratorio', 'farma conde', 'farmacenter', 'farmacia carol',
        'farmacia castro', 'farmacia cristal', 'farmacia de turno', 'farmacia estatal', 'farmacia gisselle',
        'farmacia internacional', 'farmacia nueva york', 'farmacia paris', 'farmacia san isidro', 'farmacia san roque',
        'farmacias 911', 'farmacias ahumada', 'farmacias arrocha', 'farmacias batres', 'farmacias belén',
        'farmacias benavides', 'farmacias beverley', 'farmacias bolivia', 'farmacias caridad', 'farmacias carolina y paz',
        'farmacias catedral', 'farmacias central oeste', 'farmacias chedraui', 'farmacias chávez', 'farmacias cruz azul',
        'farmacias cruz verde', 'farmacias cuxcatlecas', 'farmacias círculo', 'farmacias del ahorro', 'farmacias del dr. ahorro',
        'farmacias dr. ahorro', 'farmacias dr. simi chile', 'farmacias económicas', 'farmacias el ahorro', 'farmacias el amal',
        'farmacias el javillo', 'farmacias fischel', 'farmacias galeno', 'farmacias guadalajara', 'farmacias kielsa',
        'farmacias la bomba', 'farmacias la botica', 'farmacias la rebaja', 'farmacias la salud', 'farmacias los hidalgos',
        'farmacias machain', 'farmacias metro', 'farmacias meykos', 'farmacias peruanas', 'farmacias plaza',
        'farmacias punto farma', 'farmacias regis', 'farmacias revilla', 'farmacias saas', 'farmacias saba',
        'farmacias salemma', 'farmacias san agustín', 'farmacias san nicolás', 'farmacias san pablo', 'farmacias similares (doctor simi)',
        'farmacias simán', 'farmacias sucre', 'farmacias super farmacia', 'farmacias vicente scavone', 'farmacias ximena',
        'farmacias xolotlán', 'farmacias yza (femsa)', 'farmacity', 'farmacorp', 'farmahorro',
        'farmatodo', 'farmatodo colombia', 'fasa', 'fleni (fundación para la lucha contra las enfermedades neurológicas de la infancia)', 'fleury medicina e saúde',
        'fundación cardioinfantil (lacardio)', 'fundación oftalmológica los andes (fola)', 'fundación oftalmológica nacional', 'fundación santa fe de bogotá', 'fundación teletón (rehabilitación)',
        'fundación valle del lili', 'fybeca', 'grupo hospitalario ángeles', 'grupo pedriel / farmacias vantage', 'grupo vitamedica (laboratorios y gabinetes)',
        'hima•san pablo (servicios diagnósticos)', 'hospital albert einstein (serviços de psiquiatria)', 'hospital alcívar', 'hospital alemán', 'hospital alemão oswaldo cruz',
        'hospital auxilio mutuo', 'hospital auxilio mutuo (rehabilitación)', 'hospital auxilio mutuo (servicio de oftalmología)', 'hospital bautista (servicio de oftalmología)', 'hospital británico',
        'hospital central managua', 'hospital centro médico escalón', 'hospital centro médico hondureño', 'hospital cima san josé', 'hospital cira garcía',
        'hospital clínica bíblica', 'hospital clínica bíblica (servicio de oftalmología)', 'hospital de clínicas (servicio de oftalmología)', 'hospital de clínicas (servicios de salud mental)', 'hospital de clínicas caracas',
        'hospital de clínicas josé de san martín (servicios de salud mental)', 'hospital de diagnóstico', 'hospital de diagnóstico (servicios de neurología y rehabilitación)', 'hospital de la mujer', 'hospital de ojos club de leones',
        'hospital de ojos y oídos', 'hospital de ojos y oídos rodolfo robles', 'hospital de olhos cbv', 'hospital de salud mental federico mora', 'hospital del valle',
        'hospital del valle (laboratorios/imágenes)', 'hospital docente clínico quirúrgico hermanos ameijeiras (servicios de salud mental)', 'hospital general san juan de dios (público, de referencia nacional)', 'hospital hermanos ameijeiras', 'hospital hermilio valdizán (salud mental)',
        'hospital herrera llerandi', 'hospital herrera llerandi (servicios dentales)', 'hospital herrera llerandi (servicios diagnósticos)', 'hospital israelita albert einstein', 'hospital italiano de buenos aires',
        'hospital la católica', 'hospital la central', 'hospital metropolitano', 'hospital metropolitano (servicio de oftalmología)', 'hospital metropolitano de santiago (homs)',
        'hospital militar escuela dr. alejandro dávila bolaños (público pero con servicios privados destacados)', 'hospital moinhos de vento', 'hospital nacional', 'hospital nacional (laboratorios/imágenes)', 'hospital nacional (servicios de salud mental)',
        'hospital nacional (servicios dentales)', 'hospital nacional psiquiátrico', 'hospital nacional psiquiátrico dr. josé molina martínez', 'hospital obrero n° 1 (público, pero relevante para alta complejidad)', 'hospital odontológico docente',
        'hospital oftalmológico de brasília', 'hospital oftalmológico docente general calixto garcía', 'hospital panamericano (salud mental)', 'hospital pavia santurce', 'hospital privado hondureño',
        'hospital privado hondureño (servicios dentales)', 'hospital psicosocial', 'hospital psiquiátrico', 'hospital psiquiátrico de caracas', 'hospital psiquiátrico de la habana',
        'hospital psiquiátrico fray bernardino álvarez', 'hospital psiquiátrico lorenzo ponce', 'hospital psiquiátrico mario mendoza', 'hospital psiquiátrico san juan de dios', 'hospital punta pacífica',
        'hospital roosevelt (público/autónomo, pero importante para alta complejidad)', 'hospital sírio-libanês', 'hospital universitario austral', 'hospital universitario de caracas', 'hospital universitario esperanza',
        'hospital viera', 'hospital vilardebó (salud mental)', 'hospital vivian pellas', 'hospital vivian pellas (laboratorios/imágenes)', 'hospital vivian pellas (servicios dentales)',
        'hospital vozandes quito', 'imágenes diagnósticas de venezuela (idv)', 'ineco (instituto de neurología cognitiva)', 'inkafarma (inretail pharma)', 'inr (instituto nacional de rehabilitación)',
        'instituto cubano de oftalmología "ramón pando ferrer"', 'instituto de la visión', 'instituto de neurocirugía dr. asenjo (rehabilitación)', 'instituto de oftalmología conde de valenciana', 'instituto de psiquiatria do hcfmusp',
        'instituto espaillat cabral', 'instituto nacional de oftalmología (ino)', 'instituto nacional de rehabilitación (inr)', 'instituto zaldívar', 'integramédica (laboratorios e imágenes)',
        'interlab', 'la franco inglesa', 'laboratorio amigo', 'laboratorio borinquen', 'laboratorio central managua',
        'laboratorio cibic', 'laboratorio clínico amadita', 'laboratorio clínico biolab', 'laboratorio clínico biomédico', 'laboratorio clínico centenario',
        'laboratorio clínico centra', 'laboratorio clínico centro médico de caracas', 'laboratorio clínico colcan', 'laboratorio clínico del uruguay', 'laboratorio clínico hospital hermanos ameijeiras',
        'laboratorio clínico las américas', 'laboratorio clínico miranda', 'laboratorio clínico san pablo', 'laboratorio clínico y de patología san juan', 'laboratorio de referencia labi',
        'laboratorio gorgas (referencia nacional)', 'laboratorio labsol', 'laboratorio referencia', 'laboratorio rossi', 'laboratorio stamboulian',
        'laboratorios auna', 'laboratorios bioclinicos', 'laboratorios chopo', 'laboratorios clínicos dr. moreira', 'laboratorios clínicos echandi',
        'laboratorios clínicos raly', 'laboratorios ihss (públicos)', 'laboratorios max bloch', 'laboratorios roe', 'laboratorios san josé',
        'laboratório a+', 'laboratório hermes pardini', 'laser ocular lomas', 'locatel', 'medicacion cronica',
        'medical eye center', 'medicamentos recetados', 'medlab', 'metro pavia laboratories', 'meza dental care',
        'mifarma (inretail pharma)', 'multident', 'multident bolivia', 'médica sur', 'médica uruguaya',
        'médica uruguaya (salud mental y rehabilitación)', 'médica uruguaya (servicio de oftalmología)', 'oculaser', 'odem (organización dental mexicana)', 'odonto center',
        'odontocompany', 'odontología especializada (dr. víctor acosta)', 'odontología integral', 'odontovida', 'odontólogos asociados',
        'oftalmo centro', 'oftalmocenter', 'oftalmología integral', 'oftalmología integral de costa rica', 'oftalmología panamá',
        'oftalmología puerta del sol', 'oftalmosalud', 'oftalmosalud miraflores', 'olab (laboratorios h+)', 'optica y lentes',
        'oral sin', 'osde binario (red)', 'pague menos', 'pasteur', 'perfect smile dental',
        'pharmacy\'s', 'prevent senior (diagnóstico)', 'quest diagnostics', 'red salud uc christus', 'rede d\'or são luiz',
        'rede dor são luiz (servicios dentales)', 'rede sarah de hospitais de reabilitação', 'redsalud (laboratorios e imágenes)', 'redsalud dental', 'remedios genericos',
        'salcobrand', 'salud digna', 'sanasana (corporación gpf)', 'sanatorio americano, casmu', 'sanatorio bazterrica (salud mental)',
        'sanatorio güemes', 'sanatorio la costa', 'sanatorio mater dei', 'sanatorio migone', 'sanatorio migone (laboratorios/imágenes)',
        'sanatorio migone (servicios dentales)', 'sanatorio otamendi', 'sanatorio y hospital', 'servicio de diagnóstico por imágenes dim', 'servicio de imágenes diagnósticas (sid)',
        'servicio de oftalmología hospital de clínicas caracas', 'servicios dentales imss/issste (públicos)', 'sesion de kinesiologia', 'sesion de terapia', 'smile center honduras',
        'smile dental clinic', 'smile panamá', 'sonrident', 'sonrisa perfecta', 'sonrisa total',
        'sonrisas venezuela', 'sonría', 'sonríe uruguay', 'sorridents', 'swiss medical (diagnóstico por imágenes)',
        'swiss medical (red)', 'synlab (antes lab. clínico continental)', 'tcba (tomografía computada buenos aires)', 'tecsalud (hospital zambrano hellion / san josé)', 'teletón (rehabilitación infantil)',
        'teletón (rehabilitación)', 'teletón 2030 (rehabilitación)', 'tienda de productos naturales', 'tratamiento de ortodoncia', 'ultrafarma',
        'unidad de diagnóstico médico (udim)', 'uno salud dental', 'urgencia medica', 'vacunatorio', 'visión integral',
        'visualiza centro oftalmológico', 'vitaldent', 'vozandes (laboratorios e imágenes)', 'walgreens pr', 'zona vital',
    ],
    'Servicios': [
        'aaa (autoridad de acueductos y alcantarillados)', 'abastible', 'abono de telecomunicaciones', 'abono de telefonia movil', 'absa',
        'acodike (glp)', 'acueducto metropolitano de bucaramanga (amb)', 'aes distribuidora', 'aes dominicana (gas natural)', 'afinia',
        'agip', 'agua de puebla para todos', 'agua y drenaje de monterrey (sadm)', 'aguas andinas (región metropolitana)', 'aguas antofagasta',
        'aguas cordobesas', 'aguas santafesinas (assa)', 'air-e', 'algar telecom', 'altice rd',
        'amarilla gas', 'anda (administración nacional de acueductos y alcantarillados)', 'ande (administración nacional de electricidad)', 'antel (estatal fibra)', 'api',
        'arba', 'arbitrio de autos / permiso (suri)', 'asadas', 'at&t méxico', 'atm (agencia tributaria mendoza)',
        'axtel', 'aya (acueductos y alcantarillados)', 'aysa (agua y saneamientos argentinos)', 'aysam (aguas mendocinas)', 'bitel (viettel)',
        'brisanet (noreste)', 'caasd', 'cable & wireless (más móvil)', 'cable mágico / cablevisión local', 'cablecom',
        'caess', 'camuzzi gas del pampeno / del sur', 'ceee / equatorial', 'celerity', 'celpe',
        'celsia', 'cemig', 'cens', 'cfe (comisión federal de electricidad - cobertura nacional unificada)', 'cfe división golfo norte',
        'cfe división jalisco', 'cfe división peninsular', 'cfe internet para todos', 'cge', 'chilco (gasoil)',
        'chilquinta', 'claro', 'claro brasil', 'claro chile', 'claro colombia',
        'claro costa rica', 'claro el salvador', 'claro guatemala', 'claro honduras', 'claro nicaragua',
        'claro paraguay', 'claro perú', 'claro pr', 'claro rd', 'clesa',
        'cnel ep (guayaquil/costa)', 'cnfl', 'cnt ep', 'codelco/sistemas locales', 'coelba (neoenergia)',
        'colsecor', 'comgás', 'compagas', 'compesa', 'contribuciones de bienes raíces',
        'contribución inmobiliaria', 'contugas', 'copaco', 'copasa', 'copel',
        'coraasan', 'corpoelec (corporación eléctrica nacional)', 'corsan', 'cosemar', 'cospal',
        'cre', 'cupet (cuba petróleo)', 'cálidda', 'delapaz', 'desktop (san pablo)',
        'deusem', 'dga (dirección general de aduanas)', 'dgi (dirección general de ingresos)', 'digicel el salvador', 'digicel panamá',
        'digitel', 'disnorte-dissur', 'eaab (empresa de acueducto de bogotá)', 'ecogas', 'edea',
        'edeeste (santo domingo/este)', 'edemsa', 'eden', 'edenor', 'edenorte (santiago/cibao)',
        'edes', 'edesa', 'edeste', 'edesur', 'edesur (sur)',
        'edet', 'eegsa (empresa eléctrica de guatemala)', 'eeq (quito)', 'ejesa', 'electro oriente',
        'electro sur este', 'electrocentro', 'elfec', 'embasa', 'emcali',
        'emcali teleco', 'empire gas', 'empresa municipal de agua (empagua)', 'enacal (empresa nicaragüense de acueductos y alcantarillados sanitarios)', 'ende corporación',
        'enee (empresa nacional de energía eléctrica)', 'enel (empresa nicaragüense de electricidad)', 'enel ceará', 'enel colombia (bogotá / cundinamarca)', 'enel distribución perú',
        'enel distribución santiago', 'enel distribuição são paulo (san pablo)', 'engie méxico', 'enosa', 'ensas',
        'entel bolivia', 'entel chile', 'entel perú', 'epe', 'epec',
        'epm', 'epm (empresas públicas de medellín)', 'epm gas', 'epmaps (quito)', 'epsas (la paz)',
        'epsel', 'esph', 'essa', 'essap (empresa de servicios sanitarios del paraguay)', 'essbio',
        'esval', 'etb', 'etecsa', 'expensas extraordinarias', 'expensas ordinarias',
        'express (rosario/salta)', 'extragas', 'factura de electricidad', 'factura de gas natural', 'factura de luz',
        'finanzas cdmx', 'frontel', 'garra gas (glp)', 'gas envasado', 'gas express nieto',
        'gas nacional', 'gas nacional zeta', 'gas natural del norte', 'gas país', 'gas sayago (gas natural)',
        'gas silza', 'gas total', 'gas z', 'gasco', 'gascom',
        'gases del caribe', 'gases del occidente', 'gasmig', 'gasnor', 'gasolina total (totalenergies)',
        'gasolinera mobil', 'gasolinera petroecuador', 'gasolinera primax', 'gasolinera terpel', 'gasolinera texaco',
        'gaspro', 'gassur', 'gasvalpo', 'gigared (litoral)', 'global gas',
        'grupo tvcable (xtrim)', 'gtd manquehue / telsur', 'hidrandina', 'hidrocapital', 'hidrolara',
        'hidroven (empresas regionales)', 'hughesnet', 'hv multiplay', 'ice (instituto costarricense de electricidad)', 'ice kolbi',
        'icv (instituto de control vehicular)', 'idaan (instituto de acueductos y alcantarillados nacionales)', 'impuesto a la propiedad de bienes inmuebles (ipbi)', 'impuesto a la propiedad de vehículos automotores (ipva)', 'impuesto a la renta',
        'impuesto a las transacciones (it)', 'impuesto a las utilidades de las empresas (iue)', 'impuesto a los vehículos', 'impuesto al valor agado (iva)', 'impuesto al valor agregado (iva)',
        'impuesto de actividades económicas', 'impuesto de circulación de vehículos', 'impuesto de circulación de vehículos (placa)', 'impuesto de inmuebles', 'impuesto de propiedad de vehículos (marchamo)',
        'impuesto de rentas provincial', 'impuesto de rodamiento', 'impuesto de vehículos', 'impuesto general sobre las ventas (iva)', 'impuesto inmobiliario',
        'impuesto municipal', 'impuesto predial', 'impuesto predial unificado', 'impuesto sobre bienes inmuebles', 'impuesto sobre bienes inmuebles (ibi)',
        'impuesto sobre ingresos brutos', 'impuesto sobre ingresos personales', 'impuesto sobre inmuebles', 'impuesto sobre inmuebles urbanos', 'impuesto sobre la propiedad',
        'impuesto sobre la propiedad inmueble (crim)', 'impuesto sobre la renta', 'impuesto sobre la renta (ir)', 'impuesto sobre la renta (isr)', 'impuesto sobre vehículos automotores',
        'impuesto sobre ventas', 'impuesto vehicular', 'impuesto único sobre inmuebles (iusi)', 'inapa', 'inde (instituto nacional de electrificación)',
        'inrh (instituto nacional de recursos hidráulicos)', 'interagua (guayaquil)', 'internet por fibra optica', 'ipi (impuesto patrimonio inmobiliario)', 'iptu (imposto predial e territorial urbano)',
        'ire (impuesto a la renta empresarial)', 'irp (impuesto a la renta personal)', 'islr (impuesto sobre la renta)', 'itbms (impuesto de transferencia de bienes corporales muebles y la prestación de servicios)', 'iva (impuesto a la transferencia de bienes muebles y a la prestación de servicios)',
        'iva (impuesto al valor agregado)', 'izzi', 'jasec', 'klik / conexión digital', 'liberty costa rica',
        'liberty pr', 'light (río de janeiro)', 'lipigas', 'lipigas red', 'liquigás (copa energia)',
        'litoral gas', 'llama gas (glp)', 'llamagas', 'luma energy', 'luz del sur (lima)',
        'luz osorno', 'mantenimiento de red', 'marbete (impuesto de circulación)', 'masgas', 'matrícula vehicular',
        'megacable', 'metrogas', 'montagas', 'montevideogas (gas natural)', 'movilnet',
        'movistar', 'movistar chile', 'movistar colombia', 'movistar el salvador', 'movistar perú',
        'movistar venezuela', 'mundo', 'mundo pacífico', 'nacional gás', 'naturgy',
        'naturgy (edemet/edechi)', 'naturgy ban', 'naturgy brasil', 'naturgy méxico', 'netlife',
        'norgas', 'onat (oficina nacional de administración tributaria)', 'ose (nacional)', 'panamá gas', 'patagonia ip',
        'patente comercial y de rodados', 'patente de automotor', 'pdvsa gas', 'permiso de circulación', 'personal (telecom)',
        'personal paraguay', 'petronic', 'petropar', 'petroperú / gas natural sur', 'plan de celular',
        'progas', 'propagas (glp)', 'puma energy', 'quavii (promigas)', 'rentas córdoba',
        'riogas (glp)', 'sabesp (san pablo)', 'sacmex (sistema de aguas de la cdmx)', 'saesa', 'saguapac (santa cruz)',
        'samsa', 'sanaa (servicio autónomo nacional de acueductos y alcantarillados)', 'sanepar', 'sapal (león)', 'sar (servicio de administración de rentas)',
        'sat', 'sat (superintendencia de administración tributaria)', 'seal', 'secretaría de finanzas', 'sedacusco',
        'sedalib', 'sedapal (lima y callao)', 'sedapar', 'sefaz-mg', 'sefaz-rj',
        'sefaz-rs', 'sefaz-sc', 'sefaz-sp', 'seguro de hogar contra incendio', 'seguro de vida',
        'semapa (cochabamba)', 'seniat (servicio nacional integrado de administración aduanera y tributaria)', 'sepaf', 'servicio de agua potable', 'servicio de alarma y monitoreo',
        'servicio de cloacas y saneamiento', 'servicio de energia electrica', 'servicio de internet banda ancha', 'servicio sanitario', 'servicios de agua municipales',
        'set (subsecretaría de estado de tributación)', 'shell gas', 'siapa', 'sol gas (glp)', 'solgas',
        'sonigas', 'sri (servicio de rentas internas)', 'starlink', 'sucive', 'sulgás',
        'sumicity / giga+ (sudeste)', 'supercanal / arlink (cuyo y noa)', 'supergasbras', 'suralis (ex essal)', 'surtigas',
        'suscripcion television satelital', 't-mobile pr', 'tasa de alumbrado y limpieza', 'tasa general de inmuebles', 'tasa municipal',
        'tcc', 'telcel (américa móvil)', 'telecable', 'telecaribe / metrotel', 'telecentro (amba/pba)',
        'television por cable', 'tigo bolivia', 'tigo colombia', 'tigo costa rica', 'tigo el salvador',
        'tigo guatemala', 'tigo honduras', 'tigo nicaragua', 'tigo panamá', 'tigo paraguay',
        'tigo-une (epm)', 'tim brasil', 'tomza', 'totalenergies (glp)', 'totalgaz',
        'totalplay', 'triple a', 'tropigas', 'tropigas (glp)', 'tropigas de puerto rico',
        'ultragaz', 'une (unión eléctrica)', 'unifique (sur)', 'ute (estatal nacional)', 'vanti',
        'vero internet', 'viva', 'vivo (telefónica)', 'vtr', 'win',
        'wom', 'wom colombia', 'wow perú', 'ypf gas', 'ypfb (yacimientos petrolíferos fiscales bolivianos)',
        'zeta gas', 'zeta gas perú', 'águas do rio / cedae (río de janeiro)'
    ],
    'Transporte': [
        '1001', '99 (de didi)', 'aerobell airlines', 'aerocardal', 'aerocaribbean',
        'aeroexpresos ejecutivos', 'aerogaviota', 'aerolíneas argentinas', 'aerolíneas sosa', 'aerolíneas uruguayas (operaciones limitadas)',
        'aeromar', 'aeroméxico', 'aeroregional', 'aetra bus', 'air century',
        'air panama', 'alineacion y balanceo', 'almendrones', 'ama (autoridad de transporte metropolitano)', 'amaszonas',
        'andes líneas aéreas', 'andesmar', 'arajet', 'atsa airlines', 'autobuses (colectivos)',
        'autobuses santa fe', 'autobuses urbanos', 'autovías', 'avianca', 'avianca costa rica',
        'avianca ecuador', 'avianca el salvador', 'avianca guatemala', 'avianca honduras', 'avianca nicaragua',
        'azul linhas aéreas', 'balut', 'beat', 'berlinas del fonce', 'bicitaxis',
        'boleto de colectivo', 'boleto de tren', 'boliviana de aviación (boa)', 'bolt', 'bolívar',
        'brasilia', 'brt move (belo horizonte)', 'brt rio', 'bus norte', 'bus transit (metrobus)',
        'buser (plataforma tech)', 'buses urbanos', 'cabify', 'cambio de aceite y filtro', 'camiones/autobuses',
        'cape air', 'carga de gnc vehicular', 'carga de tarjeta transporte', 'caribe tours', 'carros públicos',
        'cata internacional', 'catarinense', 'chevallier', 'cita', 'civa',
        'cm airlines', 'coco taxis', 'colectivo / bus (tarjetas jaha / más)', 'combustible diesel gasoil', 'combustible nafta',
        'cometa', 'conchos (taxis compartidos)', 'condor bus', 'conviasa', 'conviasa (limitado)',
        'coomotor', 'cooperativa loja', 'copa airlines', 'corredores complementarios', 'costa rica green airways',
        'cot', 'cotraipbal', 'cruz del norte', 'cruz del sur', 'cubana de aviación',
        'didi', 'easy taxi', 'easyfly', 'ecojet', 'el dorado',
        'el metropolitano (brt lima)', 'eme bus', 'empresa alfaro', 'equair', 'estacion de servicio nafta',
        'estacionamiento medido', 'estacionamiento y cochera', 'estelar latinoamérica', 'estrella blanca', 'excluciva',
        'expreso bolivariano', 'expreso bávaro', 'expreso del sol', 'expreso palmira', 'expreso paraguay',
        'expresos chiricanos', 'expresos ejecutivos', 'expresso do sul', 'flecha bus', 'flota imbabura',
        'flota la macarena', 'flybondi', 'fuentes del norte', 'futura', 'general urquiza',
        'gol linhas aéreas', 'gomeria y cubiertas', 'gontijo', 'grupo ado (ado, platino, gl)', 'grupo iamsa (etn, turistar)',
        'guaguas (autobuses urbanos)', 'guaguas (colectivos)', 'guaireña', 'guanabara', 'hedman alas',
        'hugo', 'indrive', 'inspeccion tecnica vehicular', 'intertur', 'itapemirim',
        'jac santa maría', 'jetsmart argentina', 'jetsmart chile', 'jetsmart colombia', 'jetsmart perú',
        'king quality', 'la costeña', 'la preferida', 'la santaniana', 'laser airlines',
        'latam airlines (group)', 'latam airlines colombia', 'latam brasil', 'latam ecuador', 'latam paraguay (operado por otras filiales latam)',
        'latam perú', 'latam uruguay (operado por otras filiales latam)', 'lavadero de vehiculos', 'lineas', 'litegua',
        'lyft', 'línea 1 del metro de lima', 'macrobús (guadalajara)', 'magnicharters', 'mantenimiento de auto',
        'maxim', 'merval (valparaíso)', 'metro de caracas', 'metro de la cdmx', 'metro de medellín (y metrocable)',
        'metro de panamá', 'metro de quito', 'metro de santo domingo', 'metro servicios turísticos', 'metrobus (autobuses)',
        'metrobús', 'metrobús (brt)', 'metrobús (caracas)', 'metrotranvía (mendoza)', 'metrô de são paulo',
        'metrô rio', 'mexibús', 'micros', 'micros / combi (colectivos)', 'minibuses',
        'mio (cali)', 'move', 'movil bus', 'muv', 'nsa (nuestra señora de la asunción)',
        'núñez', 'oltursa', 'omnibus de méxico', 'omsa (autobuses estatales)', 'panamericana',
        'paranair', 'parche de neumatico', 'pasaje de avion', 'pasaje de omnibus', 'pasaje de subte',
        'peaje de autopista', 'peaje de ruta', 'pedidosya', 'picap', 'platinum centroamérica',
        'plusmar', 'primera plus', 'pullman bus', 'pullmantur', 'pulmitan',
        'pumakatari (bus la paz)', 'público (pisicorres)', 'rapido ochoa', 'ray', 'rebuli',
        'recarga de transporte publico', 'red de colectivos amba (tarjetas sube)', 'red metropolitana de movilidad (ex transantiago - buses y metro de santiago)', 'reina del camino', 'remises',
        'ridery', 'rodovías de venezuela', 'ruta 1 y 2', 'rutaca airlines', 'rutas del plata',
        'rysa', 'sansa airlines', 'santa', 'satena', 'seaborne airlines',
        'servicio tecnico automotor', 'servicios de bus interurbano limitados, principalmente operadores turísticos o privados.', 'sistema integrado de transporte del área metropolitana de san salvador (sitramss)', 'sitp (bogotá)', 'sky airline',
        'sky airline perú', 'sky high aviation services', 'subte de buenos aires', 'tag airlines', 'taller mecanico automotriz',
        'taxis', 'taxis colectivos', 'taxis de la compañía de turismo', 'taxis estatales', 'taxis tradicionales',
        'teleférico de santo domingo', 'teleférico la paz', 'tepsa', 'tica bus (internacional)', 'ticabus (internacional)',
        'tracopa', 'trans copacabana', 'transbarca (brt barquisimeto)', 'transmetro (barranquilla)', 'transmetro (brt)',
        'transmilenio (bogotá)', 'transnica', 'transportes aéreos guatemaltecos', 'transportes cristóbal colón', 'transportes cromotex',
        'transportes del sol', 'transportes ecuador', 'transportes la pampa', 'tren interurbano', 'tren ligero',
        'tren urbano', 'trolebús', 'trolebús y metrovía (guayaquil)', 'turbus', 'turil',
        'uber', 'union de transportistas de tocumen (utt)', 'vallarta plus', 'velotax', 'viaje en aplicacion de movilidad',
        'viaje en auto particular', 'viaje en remis', 'viaje en taxi', 'viana', 'viazul (para turistas)',
        'viação util', 'viva aerobus', 'voepass linhas aéreas', 'volaris', 'volaris el salvador',
        'vuelo de cabotaje', 'vía bariloche', 'wingo', 'yango', 'yappy taxi',
        'yummy rides', 'zenit', 'águia branca', 'ómnibus / sistema de transporte metropolitano (stm montevideo)', 'ómnibus nacionales',
        'ônibus (autobuses)'
    ],
    'Vestimenta': [
        '47 street', 'accesorios de moda', 'aishop', 'almacenes el rey', 'almacenes simán (sección ropa)',
        'anacapri', 'andesgear', 'andrea', 'animale', 'anthony\'s',
        'arezzo', 'arturo calle', 'bakers', 'bata perú', 'bayard esportes',
        'bershka', 'bosi', 'bowen', 'buzo y campera', 'c&a brasil',
        'c&a méxico', 'calzado de seguridad', 'calzado urbano', 'calzados rosario', 'calzados yumurí',
        'calzatodo', 'camila casual', 'camisa de vestir', 'camisa manga larga', 'campera de abrigo',
        'carrion', 'cartera y bolso', 'centauro', 'champs sports', 'cheeky, mimo & co.',
        'cinturon y billetera', 'clandestina', 'como quieres que te quiera', 'compra de calzado', 'corona',
        'd\'brujas', 'de prati', 'dexter', 'diunsa (sección ropa)', 'doite',
        'dudalina', 'dunkelvolk', 'el palacio de hierro', 'ela', 'ellus',
        'epa (sección ropa)', 'etafashion', 'express', 'fair play', 'falabella',
        'falabella colombia', 'fashion\'s park', 'felix b. maduro', 'fidalga (sección ropa)', 'flexi',
        'foot locker', 'forever 21', 'forum', 'galerías paseo', 'gef',
        'giannina azar', 'grimoldi', 'grupo éxito (líneas textil arkitect y bronzini)', 'guayaberas boga', 'h&m',
        'harrington', 'havaianas', 'hering', 'hites', 'inbox',
        'indian', 'indumentaria casual', 'indumentaria deportiva', 'innovasport', 'jazmín chebar',
        'jcpenney', 'journeys', 'julio', 'kosiuko', 'la curacao (sección ropa)',
        'la martina', 'lady lee', 'lippi', 'liverpool', 'lob',
        'local de indumentaria', 'local de ropa femenina', 'local de ropa masculina', 'lojas marisa', 'lojas renner',
        'lolita', 'macowens', 'macy\'s', 'macy’s (tiendas departamentales)', 'mango',
        'manzur', 'marathon peru', 'marathon sports', 'marshalls', 'martel',
        'mattelsa', 'medias y calcetines', 'melissa', 'men\'s fashion', 'moov',
        'novus', 'novus shoes', 'nueva americana', 'oechsle', 'old navy',
        'osklen', 'pacsun', 'pantalon de jean', 'pantalon de vestir', 'paris (grupo cencosud)',
        'paruolo', 'patra', 'payless shoesource', 'pernambucanas', 'pionier',
        'platanitos', 'plaza carlos iii', 'plaza lama', 'portsay', 'pull&bear',
        'punto blanco', 'rapsodia', 'remera de algodon', 'renner uruguay', 'renzo costa',
        'riachuelo', 'ricky sarkany', 'ripley', 'rm (ropa de moda)', 'ropa de invierno',
        'ropa de trabajo', 'ropa de verano', 'ropa deportiva para entrenar', 'ropa interior y lenceria', 'ropa para ninos',
        'salitre swimwear', 'scappino', 'schutz', 'sears', 'shasa',
        'siman (sección ropa)', 'snipes', 'sparta', 'sportline', 'stadium',
        'steven\'s', 'stradivarius', 'studio f', 'suburbia', 'tennis',
        'tienda de ropa', 'tiendas abaroa', 'tiendas maya', 'tiendas panamericanas', 'tiendas tijerazo (sección ropa y calzado)',
        'tiendas universal (sección ropa)', 'tj maxx', 'tl multicompras', 'topitop', 'totto',
        'traje y vestido de fiesta', 'traki (sección ropa y calzado)', 'tricot', 'tucci', 'unicentro',
        'vizzano', 'vélez', 'wanama', 'yagmour', 'zapateria',
        'zapatillas deportivas', 'zara', 'zara (grupo inditex)'
    ],
    'Vivienda': [
        'ace home center', 'adondevivir', 'afd (agencia financiera de desarrollo)', 'agencias estatales', 'agrofield',
        'aki (servicios para el hogar)', 'albañiles', 'aliss', 'almacenes boyacá', 'almacenes siman (sección hogar)',
        'alquiler de casa', 'alquiler de departamento', 'ambiente gourmet', 'arbaje soni', 'argenprop',
        'articulos de ferreteria', 'ashley furniture rd', 'ashley homestore', 'asociación la nacional de ahorros y préstamos', 'bac credomatic guatemala',
        'bac credomatic nicaragua', 'bac credomatic panamá', 'banco agrícola', 'banco atlántida', 'banco bac credomatic honduras',
        'banco bhd león', 'banco bicentenario', 'banco caja social', 'banco ciudad', 'banco continental',
        'banco cuscatlán', 'banco davivienda salvadoreño', 'banco de américa central', 'banco de chile', 'banco de costa rica',
        'banco de crédito del perú (bcp)', 'banco de crédito y comercio', 'banco de venezuela', 'banco del pacífico', 'banco del país',
        'banco do brasil', 'banco familiar', 'banco ficohsa', 'banco fie', 'banco g&t continental',
        'banco galicia', 'banco general', 'banco guayaquil', 'banco hipotecario', 'banco industrial',
        'banco itaú paraguay', 'banco lafise bancentro', 'banco macro', 'banco mercantil', 'banco mercantil santa cruz',
        'banco metropolitano', 'banco nacional de bolivia', 'banco nacional de costa rica', 'banco nación', 'banco pichincha',
        'banco popular de ahorro', 'banco popular de puerto rico', 'banco popular dominicano', 'banco promerica', 'banco provincia',
        'banco santander chile', 'banco santander uruguay', 'banco unión', 'bancoestado', 'bancolombia',
        'banistmo', 'banorte', 'banreservas', 'banrural', 'barboni',
        'barraca europa', 'barraca verrastro', 'bbva colombia', 'bbva méxico', 'bbva perú',
        'bellon', 'berríos', 'bhu (banco hipotecario del uruguay)', 'biess (banco del iess)', 'blaisten',
        'bradesco', 'brou (banco de la república oriental del uruguay)', 'c&c (casa e construção)', 'caixa econômica federal (líder absoluto en crédito inmobiliario)', 'caja de ahorros',
        'camicado (artículos de cocina y mesa)', 'cannon / la cardeuse (colchones)', 'capelli', 'casa & ideas', 'casa comfort',
        'casa ferro', 'casa paraná', 'casa y jardín', 'cemaco', 'cemaco (sección ferretería)',
        'century 21 bolivia', 'century 21 colombia', 'century 21 costa rica', 'century 21 el salvador', 'century 21 guatemala',
        'century 21 panamá', 'century 21 perú', 'century 21 rd', 'century 21 venezuela', 'cic (colchones y muebles)',
        'ciencuadras', 'clasificados el universal', 'clasificadosonline', 'classis.com.py', 'cochez y cochez',
        'colchones el dorado', 'colchones paraíso', 'colineal', 'comercial kywi', 'comercial larach',
        'comex (pinturas y recubrimientos)', 'compra de muebles para el hogar', 'compreoalquile', 'construmart', 'construmax',
        'construplaza', 'construrama', 'construred', 'construred / ferretería ochoa', 'construya y remodele (cr)',
        'corimexo', 'corotos (clasificados)', 'corredor inmobiliario', 'crate & barrel', 'cubancraigslist (sitios clasificados)',
        'cuota de credito hipotecario', 'davivienda', 'de prati (sección hogar)', 'decoracion de interiores', 'deposito de garantia alquiler',
        'dicico', 'dismac', 'distribuidora americana (diasa)', 'distribuidores de materiales de construcción locales', 'diunsa (sección ferretería)',
        'diunsa (sección hogar)', 'divino', 'doit center', 'doit center (sección hogar)', 'easy',
        'ekono', 'el colono construcción', 'el halcón', 'electricistas', 'electricistas a domicilio',
        'electricistas a domicilio cr', 'electricistas a domicilio rd', 'electricistas autorizados ve', 'electricistas calificados gt', 'electricistas certificados',
        'electricistas certificados pa', 'electricistas certificados pr', 'electricistas guayaquil', 'electricistas medellín', 'electricistas mx',
        'electricistas residenciales ni', 'electricistas urgentes', 'electricistas urgentes hn', 'elektra', 'elektra / coppel (muebles y equipamiento masivo)',
        'eletricistas', 'encanadores', 'encuentra24 (sección servicios)', 'encuentra24 costa rica', 'encuentra24 el salvador',
        'encuentra24 guatemala', 'encuentra24 honduras', 'encuentra24 nicaragua', 'encuentra24 panamá', 'engel & völkers uruguay',
        'epa (empresas polar)', 'epa (grupo ferromax)', 'etna', 'excelsior gama (sección hogar)', 'falabella / paris (sección muebles)',
        'ferretería al día', 'ferretería americana', 'ferretería epa', 'ferretería hopsa', 'ferretería industrial',
        'ferretería richardson', 'ferreterías calzada', 'ferretotal', 'ferrisariato', 'ferrocentro',
        'ferrocorp', 'ferromax', 'finca raíz', 'firstbank puerto rico', 'fondo nacional del ahorro (fna)',
        'fontenla', 'fovissste (estatales masivos)', 'gaia design', 'galleria', 'gallito luis (clasificados)',
        'getninjas', 'gonzález giménez (sección ferretería)', 'grupo ferromax', 'habitissimo', 'habitissimo perú',
        'hipermaxi (sección ferretería)', 'home center / sodimac', 'home depot', 'home depot pro referral', 'home service (claro hogar)',
        'home solutions (sodimac)', 'homecenter (sección hogar)', 'homespotter', 'homy (sodimac)', 'honorarios inmobiliaria',
        'houm', 'iguanafix', 'ikea', 'ikea (online)', 'ilumel',
        'ilumisa', 'imperial', 'importadora castro', 'infocasas bolivia', 'infocasas paraguay',
        'infocasas uruguay', 'infonavit', 'inmoweb ecuador', 'inmuebles.com', 'inmuebles24',
        'inmuebles24 guatemala', 'inmuebles24 honduras', 'inmuebles24 paraguay', 'interbank', 'itaú chile',
        'itaú unibanco', 'jamar (mueblería líder)', 'jumbo (sección hogar)', 'kitchin', 'koper furniture',
        'kywi', 'la curacao', 'la prensa gráfica (clasificados)', 'la teja (sección inmuebles)', 'la tentación',
        'lady lee', 'lamudi méxico', 'larach y cia', 'leroy merlin (sección hogar)', 'leroy merlin brasil',
        'listo & hecho', 'liverpool (sección muebles)', 'maestro', 'maestro a domicilio ec', 'maestro directo',
        'maestro perú', 'maestro.cl', 'maestros constructores', 'maestros y constructores (plataformas locales)', 'maestros y oficios py',
        'maestros y profesionales locales', 'makro (sección hogar)', 'mantenimiento de vivienda', 'mantenimiento edilicio del hogar', 'marabraz',
        'marido de aluguel', 'materiales de construccion', 'mercado libre inmuebles', 'mercado libre rd (sección inmuebles)', 'mercado libre uruguay (sección inmuebles)',
        'mercados agropecuarios (pequeñas herramientas)', 'metrocuadrado', 'mivivienda', 'mobly', 'moneyhouse',
        'morph', 'motociclo (hogar)', 'muebles capri', 'muebles dico', 'muebles expreso',
        'muebles express', 'muebles fiesta', 'muebles fina', 'muebles mary', 'mutual cartago de ahorro y préstamo',
        'national hardware', 'national lumber', 'nexo inmobiliario', 'ngo', 'nicaragua bienes raíces',
        'novey', 'oba', 'oficiosuy', 'olx brasil (sección inmuebles)', 'olx ecuador (sección inmuebles)',
        'olx honduras (sección inmuebles)', 'oriental bank', 'pago de canon locativo', 'pintura para interiores', 'plomeros',
        'plomeros 24/7 cr', 'plomeros 24/7 ni', 'plomeros 24h', 'plomeros a domicilio hn', 'plomeros a domicilio pr',
        'plomeros a domicilio ve', 'plomeros bogota', 'plomeros express gt', 'plomeros express pa', 'plomeros mx',
        'plomeros quito', 'plomeros urgentes sd', 'plomeros y electricistas asunción', 'plomeros y electricistas sv', 'plusvalia',
        'point2homes pr', 'portalinmobiliario', 'prestamo para vivienda', 'promart homecenter', 'propiedades.com',
        'quero-quero', 'quintoandar', 're/max', 're/max costa rica', 're/max méxico',
        're/max perú', 're/max puerto rico', 're/max uruguay', 'reforma certa', 'remax bolivia',
        'remax ecuador', 'remax el salvador', 'remax guatemala', 'remax honduras', 'remax nicaragua',
        'remax panamá', 'remax paraguay', 'remax rd', 'remax venezuela', 'reparacion de calefon',
        'reparacion de persianas y aberturas', 'reparos residenciais', 'revolico (sección de alquileres)', 'rodelag', 'rooms to go pr',
        'rosen', 'samaritana hardware', 'santander brasil', 'santander méxico', 'scotiabank',
        'scotiabank chile', 'scotiabank perú', 'servicio de cerrajeria', 'servicio de electricista', 'servicio de fumigacion',
        'servicio de gasista matriculado', 'servicio de limpieza domestica', 'servicio de mudanza y fletes', 'servicio de plomeria y destapaciones', 'servicios 24 horas panamá',
        'servicios de mantenimiento general', 'servicios de mantenimiento general gt', 'servicios de plomería', 'servicios de reparación cr', 'servicios del hogar py',
        'servicios estatales de mantenimiento', 'servicios múltiples rd', 'servicios reparación hogar (home depot)', 'servicios técnicos del hogar ve', 'servicios técnicos en general',
        'servicios técnicos especializados', 'servicios técnicos múltiples hn', 'servicios técnicos nicaragua', 'servicios y reparaciones montevideo', 'siman hogar',
        'sinsa', 'sinsa (sección hogar)', 'sodimac bolivia', 'sodimac homecenter (sección hogar)', 'sodimac uruguay',
        'sueñolar (colchones y muebles)', 'sukasa', 'sukasa (ferretería ligera)', 'supercasas', 'tamarindo',
        'taskrabbit', 'telhanorte', 'the home depot', 'tiendas caribe', 'tiendas de materiales de construcción (estatales)',
        'tiendas grafitti', 'tiendas landmark', 'tiendas monge', 'tiendas panamericanas', 'tiendas panorama',
        'tigo hogar', 'timbreapp', 'toctoc', 'tok&stok', 'total casa',
        'trabajadores por cuenta propia (cuentapropistas)', 'trabajos de pintura de pared', 'truper', 'tucarro.com (sección inmuebles)', 'tugó',
        'tuti', 'ultracasas', 'unicomer', 'urbania', 'vidri s.a.',
        'viva real', 'vivanuncios', 'yapo.cl (sección inmuebles)', 'zap imóveis', 'zonaprop',
    ]
}

# Limites (base) de montos realistas - puede cambiar
rangos_montos = {
    'Alimentacion': (15, 100), 'Educacion': (30, 200), 'Electrodomesticos': (200, 400), 'Inversion': (100, 300),
    'Ocio': (20, 80), 'Salud': (40, 200), 'Servicios': (20, 150), 'Transporte': (3, 200),
    'Vestimenta': (10, 300), 'Vivienda': (200, 600)
}

categorias = list(diccionario_conceptos.keys()) # Seteando las categorías
usuario_ids = np.random.randint(1, n_usuarios + 1, size=n_transacciones)

# Asignando categorias simulando comportamientos de gasto frecuentes
prob_categorias = [0.15, 0.05, 0.05, 0.10, 0.10, 0.05, 0.20, 0.10, 0.05, 0.15]
selected_categories = np.random.choice(categorias, size=n_transacciones, p=prob_categorias)

fechas_random = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 365, size=n_transacciones), unit='D')

descripciones = []
montos = []

# Novedad: Diccionario rápido para conocer el índice de poder adquisitivo de cada usuario
# Base: 1500 dólares de sueldo.
dict_indices = {row['id']: (row['ingreso_mensual'] / 1500) for _, row in df_usuarios.iterrows()}
categorias_elasticas = ['Vivienda', 'Educacion', 'Inversion', 'Ocio', 'Vestimenta', 'Electrodomesticos']

# Generación procedural de descripciones y tickets con prefijos bancarios realistas
prefijos_bancarios = ['', 'compra en ', 'pago ', 'debito ', 'transferencia a ', 'cargo por ', 'pos ', 'consumo ']
prob_prefijos = [0.35, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05, 0.05]

for user_id, cat in zip(usuario_ids, selected_categories):
    base_desc = np.random.choice(diccionario_conceptos[cat])
    prefijo = np.random.choice(prefijos_bancarios, p=prob_prefijos)
    desc = (prefijo + base_desc).strip()
    
    min_m, max_m = rangos_montos[cat]
    idx_adquisitivo = dict_indices[user_id]
    
    # Factor de escalado dinámico según sueldo y elasticidad
    if cat in categorias_elasticas:
        max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.5)
        min_m = min_m * (1 + (idx_adquisitivo - 1) * 0.2)
    else:
        max_m = max_m * (1 + (idx_adquisitivo - 1) * 0.2)
    
    # Distribución Triangular
    if cat in ['Servicios', 'Educacion', 'Vivienda']:
        moda_m = min_m + (max_m - min_m) * 0.5
    else:
        moda_m = min_m + (max_m - min_m) * 0.25
    monto = round(np.random.triangular(min_m, moda_m, max_m), 2)
    
    descripciones.append(desc)
    montos.append(monto)

# Ruido de etiquetado desactivado
# ruido_mask = np.random.rand(n_transacciones) < 0.10
# categorias_ruido = np.random.choice(categorias, size=ruido_mask.sum())
# selected_categories[ruido_mask] = categorias_ruido

df_transacciones = pd.DataFrame({
    'id': range(1, n_transacciones + 1),
    'usuario_id': usuario_ids,
    'descripcion': descripciones,
    'valor': montos,
    'categoria': selected_categories,
    'fecha': fechas_random.strftime('%Y-%m-%d')
})

print("Transacciones simuladas:", df_transacciones.shape)


Transacciones simuladas: (240000, 6)


In [4]:
pd.Series({k: len(v) for k, v in diccionario_conceptos.items()}).sort_values(ascending=False)

Salud                510
Vivienda             420
Servicios            413
Educacion            352
Ocio                 310
Transporte           286
Inversion            186
Vestimenta           183
Alimentacion         165
Electrodomesticos    136
dtype: int64

### 2.1 Revisión de transacciones


In [5]:
# Para ver cuántas transacciones promedio tiene cada usuario
tx_por_usuario = df_transacciones.groupby('usuario_id').size().reset_index(name='tx_anuales')
tx_por_usuario['tx_mensuales'] = tx_por_usuario['tx_anuales'] / 12.0

print("Transacciones por usuario:")
display(tx_por_usuario[['tx_anuales', 'tx_mensuales']].mean())

Transacciones por usuario:


tx_anuales      133.333333
tx_mensuales     11.111111
dtype: float64

## 3. Perfil Financiero - Revisar umbrales. Discutir en general
Auditoría de 4 pasos sin bucles

`La regla bancaria internacional (28/36). Se eligieron umbrales "bajos y apretados" (22% y 26%) para forzar a que la distribución poblacional quedara repartida aproximadamente en un 40/35/25, permitiendo un entrenamiento de machine learning estable y estadísticamente sano.`

> **Aislar los gastos fijos (Vivienda y Servicios):**
> El algoritmo busca todos los pagos de alquiler, luz o agua de un usuario y saca el promedio de cuánto paga al mes. Supongamos que el usuario "Juan" paga un promedio de $600 mensuales por estos conceptos. 

> **Ratio de endeudamiento:**
> Toma esos $600 y los divide por su sueldo. El cálculo arroja un 40%. Eso significa que Juan compromete el 40% de su sueldo solo para sobrevivir mes a mes.

> **Calcular el hábito de ahorro:**
> El algoritmo cuenta cuántos "tickets de compra" de Juan pertenecen a la categoría "Inversión" a lo largo del año. Si Juan tiene cero tickets de inversión, se le cataloga con Frecuencia de Ahorro "Ninguna".

---

##### **Armado del perfil financiero:** Cruza el endeudamiento con el ahorro de la siguiente manera: 
> * Si el endeudamiento de la persona es mayor al 26% de su sueldo, automáticamente se etiqueta como 'En Riesgo' (la persona está ahogada financieramente).
> * Si su endeudamiento es bajo (menor al 22%) Y además tiene un hábito de ahorro Medio o Alto, se corona como 'Saludable'.
> * Cualquier combinación intermedia o gris (ejemplo, tiene poco endeudamiento pero nunca invierte un centavo), cae en la categoría 'En Observación'.



* Se calculó el promedio mensual (mean()) del gasto en categorías fijas (Vivienda y Servicios).
* Se obtuvo el ratio de endeudamiento dividiendo dicho promedio por el ingreso mensual del usuario.
* Se contabilizó la frecuencia de transacciones en la categoría "Inversión" para definir la categoría de ahorro.
* Se aplicó np.select() para asignar el perfil financiero según umbrales de endeudamiento (22% y 26%) y frecuencia de ahorro.

In [6]:
# 1. Promedio mensual de gastos fijos (Vivienda y Servicios)
gastos_fijos = df_transacciones[df_transacciones['categoria'].isin(['Vivienda', 'Servicios'])]
gastos_fijos_promedio = gastos_fijos.groupby(['usuario_id', 'categoria'])['valor'].mean().unstack(fill_value=0)
if 'Vivienda' not in gastos_fijos_promedio.columns: gastos_fijos_promedio['Vivienda'] = 0
if 'Servicios' not in gastos_fijos_promedio.columns: gastos_fijos_promedio['Servicios'] = 0
gastos_fijos_promedio['gasto_fijo_total'] = gastos_fijos_promedio['Vivienda'] + gastos_fijos_promedio['Servicios']

# 2. Conteo de inversiones
inversiones = df_transacciones[df_transacciones['categoria'] == 'Inversion']
conteo_inversiones = inversiones.groupby('usuario_id').size().reset_index(name='n_inversion_anual')

# 3. Merge con usuarios
df_calc = df_usuarios[['id', 'ingreso_mensual']].copy()
df_calc = df_calc.merge(gastos_fijos_promedio[['gasto_fijo_total']], left_on='id', right_on='usuario_id', how='left').fillna(0)
df_calc = df_calc.merge(conteo_inversiones, left_on='id', right_on='usuario_id', how='left').fillna(0)

# 4. Cálculos vectorizados
df_calc['nivel_endeudamiento'] = np.round((df_calc['gasto_fijo_total'] / df_calc['ingreso_mensual']) * 100, 2)
df_calc['nivel_endeudamiento'] = df_calc['nivel_endeudamiento'].clip(5.0, 90.0)

df_calc['inversiones_mensuales'] = df_calc['n_inversion_anual'] / 12.0

# Asignar frecuencia de ahorro
condiciones_ahorro = [
    df_calc['inversiones_mensuales'] == 0,
    df_calc['inversiones_mensuales'] < 1.0,
    df_calc['inversiones_mensuales'] < 3.0
]
opciones_ahorro = ['Ninguna', 'Baja', 'Media']
df_calc['frecuencia_ahorro'] = np.select(condiciones_ahorro, opciones_ahorro, default='Alta')

# Asignar perfil financiero
condiciones_perfil = [
    df_calc['nivel_endeudamiento'] > 26.0,
    (df_calc['nivel_endeudamiento'] > 22.0) | (df_calc['frecuencia_ahorro'] == 'Ninguna'),
    (df_calc['nivel_endeudamiento'] <= 22.0) & (df_calc['frecuencia_ahorro'].isin(['Media', 'Alta']))
]
opciones_perfil = ['En riesgo', 'En observacion', 'Saludable']
df_calc['perfil_financiero'] = np.select(condiciones_perfil, opciones_perfil, default='En observacion')

# Integrar resultados al dataframe original
df_usuarios = pd.merge(df_usuarios, df_calc[['id', 'nivel_endeudamiento', 'frecuencia_ahorro', 'perfil_financiero']], on='id')

print("Perfiles consistentes calculados. Distribución:")
print(df_usuarios['perfil_financiero'].value_counts(normalize=True) * 100)


Perfiles consistentes calculados. Distribución:
perfil_financiero
En observacion    42.777778
Saludable         35.888889
En riesgo         21.333333
Name: proportion, dtype: float64


## 4. Prevención de Data Leakage - Prueba
Prevención de fugas:
*   **Para Perfiles (Usuarios):** Corte transversal común y corriente.
*   **Para Transacciones (NLP):** Corte temporal. El modelo entrena con el historial de enero-agosto y se evalúa de manera ciega sobre los gastos del futuro (Nov-Dic).

---

Train (60%): Exclusivo para ajustar los parámetros de los modelos.

Validation (20%): Entorno de prueba intermedio. Permite comparar diferentes algoritmos (ej. Regresión Logística vs. Árboles), probar hiperparámetros y detectar sobreajuste sin tocar el examen final.

Test (20%): Queda completamente sellado e intocado durante la etapa de desarrollo. Solo se evalúa una única vez al final de todo el proyecto para obtener la métrica definitiva de rendimiento sin ningún sesgo de tuning.

In [7]:
# 1. Split Transversal Agrupado (Group Holdout Split) por ID de Usuario
splits_usr = np.random.choice(['train', 'val', 'test'], size=n_usuarios, p=[0.6, 0.2, 0.2])
df_usuarios['split'] = splits_usr

# 2. Split Temporal (Out-of-Time) para Transacciones
# Enero a Agosto (train), Sept-Oct (val), Nov-Dic (test)
meses = pd.to_datetime(df_transacciones['fecha']).dt.month
condiciones = [
    meses <= 8,
    meses.isin([9, 10]),
    meses >= 11
]
df_transacciones['split'] = np.select(condiciones, ['train', 'val', 'test'], default='test')

print("Distribución Transversal de Usuarios (Cross-sectional):")
print(df_usuarios['split'].value_counts(normalize=True) * 100)
print("\nDistribución Temporal de Transacciones (Out-of-Time):")
print(df_transacciones['split'].value_counts(normalize=True) * 100)

Distribución Transversal de Usuarios (Cross-sectional):
split
train    62.055556
test     20.222222
val      17.722222
Name: proportion, dtype: float64

Distribución Temporal de Transacciones (Out-of-Time):
split
train    66.45875
val      16.79250
test     16.74875
Name: proportion, dtype: float64


## 5. Exportación de Archivos Semilla
Exportamos tablas robustas y finalizadas a `data/`.

In [8]:
os.makedirs('data', exist_ok=True)

# 1. Exportación Cruda (Uso Interno de Data Science)
# Estos archivos mantienen la columna "split" esencial para EDA y Modelado.
df_usuarios.to_csv('data/usuarios.csv', index=False)
df_transacciones.to_csv('data/transacciones.csv', index=False)
df_usuarios.to_json('data/usuarios.json', orient='records', indent=2, force_ascii=False)
df_transacciones.to_json('data/transacciones.json', orient='records', indent=2, force_ascii=False)

# 2. Exportación Limpia (Uso Externo para Backend)
# Se remueve "split" para evitar problemas de compatibilidad en Java.
df_usuarios.drop(columns=['split']).to_csv('data/usuarios_backend.csv', index=False)
df_transacciones.drop(columns=['split']).to_csv('data/transacciones_backend.csv', index=False)
df_usuarios.drop(columns=['split']).to_json('data/usuarios_backend.json', orient='records', indent=2, force_ascii=False)
df_transacciones.drop(columns=['split']).to_json('data/transacciones_backend.json', orient='records', indent=2, force_ascii=False)

print("¡Exportación exitosa! Semillas crudas y limpias generadas correctamente.")


¡Exportación exitosa! Semillas crudas y limpias generadas correctamente.


In [9]:
# check rápido
df_usuarios.head()

,id,nombre,ingreso_mensual,nivel_endeudamiento,frecuencia_ahorro,perfil_financiero,split
0,1,Feliciana,2810.89,21.90,Baja,En observacion,val
1,2,Dani,4827.50,19.74,Media,Saludable,val
2,3,Amador,4061.98,20.65,Media,Saludable,train
3,4,Manuela,3595.30,22.54,Media,En observacion,val
4,5,Leonardo,2046.07,27.04,Baja,En riesgo,train
